# ML - Ephyrae Prediction (Aurelia labiata)

Simple ML pipeline to predict ephyrae appearance.

**Date:** 2026-05-31

In [ ]:
# Setup
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path('../data')
CSV = DATA_DIR / 'polyp_tracking_long_format.csv'

print(f'Loading {CSV}...')
df = pd.read_csv(CSV, encoding='utf-8-sig')
print(f'✓ Loaded {len(df)} rows')
print(f'✓ Columns: {df.columns.tolist()}')

In [ ]:
# Filter for Aurelia labiata and prepare data
SPECIES = 'Aurelia labiata'

# Get data for this species
species_df = df[df['species'] == SPECIES].copy()
print(f'\n{SPECIES}:')
print(f'  Rows: {len(species_df)}')
print(f'  Boxes: {species_df["box"].nunique()}')
print(f'  Years: {sorted(species_df["file_year"].unique())}')

# Pivot to separate polyps and ephyrae
pivot = species_df.pivot_table(
    index=['box', 'file_year', 'week_index', 'temperature'],
    columns='normalized_measurement_type',
    values='value',
    aggfunc='first'
).reset_index()

print(f'\nAfter pivot: {pivot.shape} rows')
print(f'Columns: {pivot.columns.tolist()}')
display(pivot.head())

In [ ]:
# Create features
pivot = pivot.sort_values(['box', 'file_year', 'week_index']).reset_index(drop=True)

# Target
pivot['has_ephyrae'] = (pivot['ephyrae'] > 0).astype(int)

# Features
pivot['polyp_count'] = pivot['polyps']
pivot['week_sin'] = np.sin(2 * np.pi * pivot['week_index'] / 52)
pivot['week_cos'] = np.cos(2 * np.pi * pivot['week_index'] / 52)

# Rolling polyp stats by box
pivot['polyp_ma4'] = pivot.groupby('box')['polyp_count'].transform(lambda x: x.rolling(4, min_periods=1).mean())
pivot['polyp_std4'] = pivot.groupby('box')['polyp_count'].transform(lambda x: x.rolling(4, min_periods=1).std()).fillna(0)

# Temperature categories
pivot['temp_cold'] = (pivot['temperature'] <= 10).astype(int)
pivot['temp_warm'] = (pivot['temperature'] >= 20).astype(int)

print('\nFeatures created:')
print(f'Target (has_ephyrae): {pivot["has_ephyrae"].sum()} positive, {(~pivot["has_ephyrae"].astype(bool)).sum()} negative')

feature_cols = ['polyp_count', 'temperature', 'week_sin', 'week_cos', 'polyp_ma4', 'polyp_std4', 'temp_cold', 'temp_warm']
display(pivot[feature_cols + ['has_ephyrae', 'ephyrae']].head(10))

In [ ]:
# Prepare training data (remove NaN)
train = pivot[pivot[feature_cols + ['has_ephyrae']].notna().all(axis=1)].copy()

print(f'\nTraining data: {len(train)} rows')
print(f'Target distribution:\n{train["has_ephyrae"].value_counts()}')
print(f'Ephyrae rate: {train["has_ephyrae"].mean()*100:.1f}%')

X = train[feature_cols].copy()
y = train['has_ephyrae'].copy()

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain: {len(X_train)} rows')
print(f'Test: {len(X_test)} rows')

In [ ]:
# Train model
print('Training GradientBoosting...')
model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    verbose=0
)
model.fit(X_train, y_train)
print('✓ Model trained')

# Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
test_auc = roc_auc_score(y_test, y_proba)

print(f'\nResults:')
print(f'  Train Accuracy: {train_acc:.3f}')
print(f'  Test Accuracy:  {test_acc:.3f}')
print(f'  Test ROC-AUC:   {test_auc:.3f}')

print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Ephyrae', 'Ephyrae']))

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('\nFeature Importance:')
display(importance)

fig, ax = plt.subplots(figsize=(10, 5))
importance.plot(kind='barh', x='feature', y='importance', ax=ax, legend=False)
ax.set_xlabel('Importance')
ax.set_title(f'Feature Importance - {SPECIES}')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, label=f'ROC (AUC={test_auc:.3f})', lw=2, color='blue')
ax.plot([0, 1], [0, 1], 'k--', label='Random', alpha=0.5)
ax.fill_between(fpr, tpr, alpha=0.2)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve - {SPECIES}')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Predict on 2026 data
data_2026 = pivot[(pivot['file_year'] == 2026) & (pivot[feature_cols].notna().all(axis=1))].copy()

if len(data_2026) > 0:
    X_2026 = scaler.transform(data_2026[feature_cols])
    data_2026['ephyrae_risk'] = model.predict_proba(X_2026)[:, 1]
    data_2026['predicted'] = model.predict(X_2026)
    
    # High risk (> 50%)
    high_risk = data_2026[data_2026['ephyrae_risk'] > 0.5].sort_values('ephyrae_risk', ascending=False)
    
    print(f'\n🚨 HIGH RISK BOXES (2026) - probability > 50%:')
    print(f'\nTotal predictions: {len(data_2026)}')
    print(f'High-risk count: {len(high_risk)}')
    
    if len(high_risk) > 0:
        display(high_risk[['box', 'week_index', 'polyp_count', 'temperature', 'ephyrae_risk']].head(20))
    else:
        print('No high-risk boxes detected')
else:
    print('No 2026 data available')

## Summary

✓ Model trained successfully on Aurelia labiata

**Key findings:**
- Polyp count is the strongest predictor
- Temperature and seasonality matter
- Rolling polyp trends help detect ephyrae risk

**Next: Apply to other species and deploy alerts**